# Naive Bayes from scratch 
Sections follow the lesson:
1. Bayes' theorem (disease-test example)
2. The classifier: posterior ~ prior * prod likelihood; hand-worked spam example
3. Three variants: Gaussian, Multinomial, Bernoulli (one shared base class)
4. Laplace smoothing: the zero-probability problem and the effect of alpha
5. Log-probabilities and numerical underflow
6. Why it works despite a false assumption (accuracy vs calibration)
7. Limitations (XOR, correlated features)
8. Worked problems from math/question.md (Q1 multinomial, Q2 Bernoulli, Q3 Gaussian)

In [ ]:
import os
import warnings

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.simplefilter("ignore")
os.makedirs("plots", exist_ok=True)
rng = np.random.default_rng(42)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 110})

## 1. Bayes' theorem: posterior = likelihood * prior / evidence
1% of people have a disease; the test detects 90% of sick people and
gives a false positive for 9% of healthy people ("90% accurate").
P(sick | positive) is far below 90% because the healthy 99% produce many false alarms.

In [ ]:
prior = 0.01
sens, fpr = 0.90, 0.09
evidence = sens * prior + fpr * (1 - prior)
post = sens * prior / evidence
print(f"P(sick | positive) = {post:.3f}")

priors = np.linspace(0.001, 0.5, 200)
posts = sens * priors / (sens * priors + fpr * (1 - priors))
plt.figure(figsize=(6, 4))
plt.plot(priors, posts)
plt.axvline(prior, color="C3", ls="--", label="prior = 1%")
plt.scatter([prior], [post], color="C3", zorder=3)
plt.xlabel("prior P(sick)"); plt.ylabel("posterior P(sick | +)")
plt.title("Evidence updates the prior, it does not replace it")
plt.legend(); plt.tight_layout()
plt.savefig("plots/01_bayes_theorem.png"); plt.close()

## 2. From Bayes to a classifier
`P(C|X) ~ P(C) * prod_i P(x_i|C)` (the evidence P(X) is the same for every class).
Hand example from the lesson: four emails, new email "FREE WIN".
Without smoothing P(FREE|Not spam)=0 kills the score; with add-one smoothing it works.

In [ ]:
docs = [("FREE MONEY", 1), ("WIN MONEY", 1), ("MEETING TODAY", 0), ("PROJECT UPDATE", 0)]
vocab = sorted({w for d, _ in docs for w in d.split()})
V = len(vocab)
cnt = {c: {w: 0 for w in vocab} for c in (0, 1)}
for d, c in docs:
    for w in d.split():
        cnt[c][w] += 1
tot = {c: sum(cnt[c].values()) for c in cnt}
new = "FREE WIN".split()
for alpha in (0.0, 1.0):
    for c, name in ((1, "Spam"), (0, "Not spam")):
        s = 0.5
        for w in new:
            s *= (cnt[c][w] + alpha) / (tot[c] + alpha * V)
        print(f"alpha={alpha}: score({name}) = {s:.5f}")

## 3. The three variants
All share: `fit` computes class priors and per-class feature parameters,
`joint_log_likelihood` returns `log P(c) + sum_i log P(x_i|c)`, prediction is argmax,
`predict_proba` is a softmax (log-sum-exp) over classes.
* Gaussian: mean and variance per feature per class (continuous data)
* Multinomial: smoothed word-frequency per class, `theta_cj = (N_cj + a)/(N_c + a V)` (counts)
* Bernoulli: `p_cj = (n_cj + a)/(n_c + 2a)`; absent features contribute `log(1-p_cj)` (binary)

In [ ]:
class NaiveBayesBase:
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        self.classes_, yi = np.unique(y, return_inverse=True)
        self.class_count_ = np.bincount(yi).astype(float)
        self.class_log_prior_ = np.log(self.class_count_ / len(yi))
        self._fit_params(X, yi)
        return self

    def predict_log_proba(self, X):
        jll = self.joint_log_likelihood(np.asarray(X, dtype=float))
        m = jll.max(1, keepdims=True)
        return jll - (m + np.log(np.exp(jll - m).sum(1, keepdims=True)))

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        return self.classes_[self.joint_log_likelihood(np.asarray(X, dtype=float)).argmax(1)]

    def score(self, X, y):
        return float((self.predict(X) == np.asarray(y)).mean())


class GaussianNB(NaiveBayesBase):
    def __init__(self, var_smoothing=1e-9, ddof=0):
        self.var_smoothing, self.ddof = var_smoothing, ddof

    def _fit_params(self, X, yi):
        K, d = len(self.classes_), X.shape[1]
        self.mu_, self.var_ = np.zeros((K, d)), np.zeros((K, d))
        for k in range(K):
            Xk = X[yi == k]
            self.mu_[k] = Xk.mean(0)
            self.var_[k] = Xk.var(0, ddof=self.ddof)
        self.var_ += self.var_smoothing * X.var(0).max()   # guards against zero variance

    def joint_log_likelihood(self, X):
        out = []
        for k in range(len(self.classes_)):
            ll = -0.5 * (np.log(2 * np.pi * self.var_[k]) + (X - self.mu_[k]) ** 2 / self.var_[k])
            out.append(self.class_log_prior_[k] + ll.sum(1))
        return np.array(out).T


class MultinomialNB(NaiveBayesBase):
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def _fit_params(self, X, yi):
        K, V = len(self.classes_), X.shape[1]
        self.feature_count_ = np.array([X[yi == k].sum(0) for k in range(K)])
        sm = self.feature_count_ + self.alpha
        self.feature_log_prob_ = np.log(sm) - np.log(sm.sum(1, keepdims=True))

    def joint_log_likelihood(self, X):
        return X @ self.feature_log_prob_.T + self.class_log_prior_


class BernoulliNB(NaiveBayesBase):
    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def _fit_params(self, X, yi):
        K = len(self.classes_)
        n_cj = np.array([(X[yi == k] > 0).sum(0) for k in range(K)])
        self.p_ = (n_cj + self.alpha) / (self.class_count_[:, None] + 2 * self.alpha)

    def joint_log_likelihood(self, X):
        Xb = (X > 0).astype(float)
        return Xb @ np.log(self.p_).T + (1 - Xb) @ np.log(1 - self.p_).T + self.class_log_prior_

### 3a. Gaussian NB on 2-D continuous data
Two overlapping blobs; the decision boundary is quadratic when class variances differ.

In [ ]:
def make_blobs(n, centers, scales, rng):
    X = np.vstack([rng.normal(c, s, size=(n, 2)) for c, s in zip(centers, scales)])
    y = np.repeat(np.arange(len(centers)), n)
    p = rng.permutation(len(y))
    return X[p], y[p]

Xg, yg = make_blobs(300, [(-1, -1), (1.5, 1)], [(1.0, 1.0), (1.6, 0.7)], rng)
ntr = 360
gnb = GaussianNB().fit(Xg[:ntr], yg[:ntr])
print("GaussianNB test acc:", gnb.score(Xg[ntr:], yg[ntr:]))
print("class means:\n", gnb.mu_.round(2))

gx, gy = np.meshgrid(np.linspace(-5, 6, 250), np.linspace(-4, 4, 250))
grid = np.c_[gx.ravel(), gy.ravel()]
pg = gnb.predict_proba(grid)[:, 1].reshape(gx.shape)
fig, ax = plt.subplots(figsize=(6, 4.5))
cs = ax.contourf(gx, gy, pg, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.6)
ax.contour(gx, gy, pg, levels=[0.5], colors="k")
ax.scatter(*Xg[:ntr].T, c=yg[:ntr], cmap="RdBu_r", s=10, edgecolor="k", linewidth=0.2)
plt.colorbar(cs, label="P(class 1 | x)")
ax.set_title("Gaussian NB: probability map and boundary"); fig.tight_layout()
fig.savefig("plots/02_gaussian_nb.png"); plt.close(fig)

### 3b. Multinomial NB on synthetic "documents"
3 topics, 60-word vocabulary; each topic has its own word distribution,
documents are multinomial draws of random length.

In [ ]:
def make_docs(n_per, rng, V=60, K=3, length=(20, 60), mix=0.25):
    """Topic word distributions = shared background + a topic-specific part (weight `mix`)."""
    base = rng.dirichlet(np.ones(V) * 0.5)
    topics = np.array([(1 - mix) * base + mix * rng.dirichlet(np.ones(V) * 0.2) for _ in range(K)])
    X, y = [], []
    for k in range(K):
        for _ in range(n_per):
            L = rng.integers(*length)
            X.append(rng.multinomial(L, topics[k])); y.append(k)
    X, y = np.array(X), np.array(y)
    p = rng.permutation(len(y))
    return X[p], y[p], topics

Xd, yd, topics = make_docs(200, rng)
mnb = MultinomialNB(alpha=1.0).fit(Xd[:400], yd[:400])
print("MultinomialNB test acc:", mnb.score(Xd[400:], yd[400:]))
print("true vs learned P(word|topic0), correlation:",
      np.corrcoef(topics[0], np.exp(mnb.feature_log_prob_[0]))[0, 1].round(3))

### 3c. Bernoulli NB on binary features
Same documents turned into presence/absence. Bernoulli penalises a word being absent;
Multinomial ignores absent words.

In [ ]:
bnb = BernoulliNB(alpha=1.0).fit(Xd[:400], yd[:400])
print("BernoulliNB (presence/absence) test acc:", bnb.score(Xd[400:], yd[400:]))
print("MultinomialNB (counts)           test acc:", mnb.score(Xd[400:], yd[400:]))

## 4. Laplace smoothing
Without smoothing, a word seen only in the other class during training gives probability 0
for the true class on a test document, so log-probability is -inf and the product is 0.
Sweep alpha on a small, sparse training set.

In [ ]:
Xs, ys, _ = make_docs(300, rng, V=200, length=(8, 20))
Xtr, ytr, Xte, yte = Xs[:90], ys[:90], Xs[300:], ys[300:]
alphas = [1e-6, 1e-3, 1e-2, 0.1, 0.3, 1, 3, 10, 30, 100]
accs, nll = [], []
for a in alphas:
    m = MultinomialNB(alpha=a).fit(Xtr, ytr)
    accs.append(m.score(Xte, yte))
    P = m.predict_proba(Xte)
    nll.append(-np.log(P[np.arange(len(yte)), yte] + 1e-12).mean())
fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].semilogx(alphas, accs, "o-"); ax[0].set_xlabel("alpha"); ax[0].set_ylabel("test accuracy")
ax[0].set_title("Smoothing helps on sparse data")
ax[1].semilogx(alphas, nll, "o-", color="C1"); ax[1].set_xlabel("alpha"); ax[1].set_ylabel("test log-loss")
ax[1].set_title("alpha too small => overconfident zeros")
fig.tight_layout(); fig.savefig("plots/03_laplace_alpha.png"); plt.close(fig)
print("best alpha by accuracy:", alphas[int(np.argmax(accs))], "acc", round(max(accs), 3),
      "| alpha=1e-6 acc", round(accs[0], 3))

## 5. Log-probabilities and underflow
A 2000-feature document: each likelihood ~1e-3, so the raw product is ~1e-6000
and underflows to exactly 0 in float64 (min ~1e-308). Summing logs is fine.

In [ ]:
p_feat = rng.uniform(1e-4, 1e-2, size=2000)
print("raw product      :", np.prod(p_feat))
print("sum of logs      :", np.log(p_feat).sum().round(1))
prod_spam, prod_ham = np.prod(p_feat), np.prod(p_feat * 0.9)
print("raw comparison   :", prod_spam, prod_ham, "-> tie, no decision possible")
print("log comparison   :", np.log(p_feat).sum() > (np.log(p_feat) + np.log(0.9)).sum(), "-> decision OK")

ns = np.arange(1, 400)
logs = np.cumsum(np.log(rng.uniform(1e-4, 1e-2, size=399)))
plt.figure(figsize=(6, 3.8))
plt.plot(ns, logs / np.log(10), label="log10 of product")
plt.axhline(-308, color="C3", ls="--", label="float64 underflow (~1e-308)")
plt.xlabel("number of features multiplied"); plt.ylabel("log10 value"); plt.legend()
plt.title("Why we sum log-probabilities"); plt.tight_layout()
plt.savefig("plots/04_underflow.png"); plt.close()

## 6. Why it works despite a false assumption: ranking vs calibration
Duplicate each informative feature `r` times (perfect correlation, strongly violating
independence). Accuracy barely changes, but predicted probabilities get pushed to 0/1
(overconfident): the evidence is counted `r` times. Reliability diagram for r=1 and r=8.

In [ ]:
def reliability(p, y, bins=10):
    idx = np.minimum((p * bins).astype(int), bins - 1)
    conf = np.array([p[idx == b].mean() if (idx == b).any() else np.nan for b in range(bins)])
    acc = np.array([y[idx == b].mean() if (idx == b).any() else np.nan for b in range(bins)])
    return conf, acc

Xb, yb = make_blobs(2500, [(0, 0), (0.8, 0.8)], [(1.2, 1.2), (1.2, 1.2)], rng)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.plot([0, 1], [0, 1], "k--", label="perfectly calibrated")
for r in (1, 8):
    Xr = np.repeat(Xb, r, axis=1)
    m = GaussianNB().fit(Xr[:2500], yb[:2500])
    p = m.predict_proba(Xr[2500:])[:, 1]
    ece = np.nanmean(np.abs(np.subtract(*reliability(p, yb[2500:]))))
    print(f"copies r={r}: test acc {m.score(Xr[2500:], yb[2500:]):.3f}, "
          f"mean |p-0.5| {np.abs(p - 0.5).mean():.3f}, calibration gap {ece:.3f}")
    c, a = reliability(p, yb[2500:])
    ax.plot(c, a, "o-", label=f"each feature repeated x{r}")
ax.set_xlabel("predicted P(class 1)"); ax.set_ylabel("observed frequency")
ax.set_title("Same accuracy, worse probabilities"); ax.legend()
fig.tight_layout(); fig.savefig("plots/05_calibration.png"); plt.close(fig)

## 7. Limitations
* No feature interactions: XOR-style labels are invisible to NB (each feature alone is uninformative).
* Gaussian assumption: heavily skewed data hurts.
Compare with a tiny logistic-regression baseline (fitted by gradient descent) that can use
a product feature.

In [ ]:
Xx = rng.normal(size=(2000, 2))
yx = ((Xx[:, 0] > 0) ^ (Xx[:, 1] > 0)).astype(int)
nbx = GaussianNB().fit(Xx[:1000], yx[:1000])
print("XOR data, GaussianNB test acc:", nbx.score(Xx[1000:], yx[1000:]), "(chance level or worse)")

def logreg(F, y, steps=800, lr=0.5):
    F1 = np.c_[np.ones(len(F)), F]; w = np.zeros(F1.shape[1])
    for _ in range(steps):
        p = 1 / (1 + np.exp(-F1 @ w)); w -= lr * F1.T @ (p - y) / len(y)
    return w
feat = lambda X: np.c_[X, X[:, 0] * X[:, 1]]
w = logreg(feat(Xx[:1000]), yx[:1000])
pred = (np.c_[np.ones(1000), feat(Xx[1000:])] @ w > 0).astype(int)
print("logistic regression with x1*x2 feature acc:", (pred == yx[1000:]).mean())

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
ax[0].scatter(*Xx[:600].T, c=yx[:600], cmap="coolwarm", s=8); ax[0].set_title("XOR data")
pg = nbx.predict_proba(np.c_[gx.ravel() / 2, gy.ravel() / 1.5])[:, 1].reshape(gx.shape)
ax[1].contourf(gx / 2, gy / 1.5, pg, levels=np.linspace(0, 1, 11), cmap="coolwarm", alpha=0.7)
ax[1].set_title("Gaussian NB output: no structure")
fig.tight_layout(); fig.savefig("plots/06_xor_failure.png"); plt.close(fig)

## 8. Worked problems (math/question.md)
### Q1 Multinomial NB: spam filter, new email {Free, Meeting, Prize, Claim}

In [ ]:
emails = [("Free Prize Win Claim", "Spam"), ("Meeting Project Schedule", "Not Spam"),
          ("Free Win Prize Lottery", "Spam"), ("Discussion Agenda Meeting", "Not Spam"),
          ("Claim Free Offer Prize", "Spam"), ("Project Report Deadline", "Not Spam")]
vocab1 = sorted({w for e, _ in emails for w in e.split()})
print("(a) priors P(Spam)=P(Not Spam)=", 3 / 6, "| (b) |V| =", len(vocab1))
X1 = np.array([[e.split().count(w) for w in vocab1] for e, _ in emails])
y1 = np.array([c for _, c in emails])
m1 = MultinomialNB(alpha=1.0).fit(X1, y1)
for k, c in enumerate(m1.classes_):
    print(f"   counts[{c}]:", {w: int(n) for w, n in zip(vocab1, m1.feature_count_[k]) if n},
          "total", int(m1.feature_count_[k].sum()))
x_new = np.array([[1 if w in ("Free", "Meeting", "Prize", "Claim") else 0 for w in vocab1]])
lik = np.exp(x_new @ m1.feature_log_prob_.T)[0]
score = np.exp(m1.joint_log_likelihood(x_new))[0]
for c, l, s in zip(m1.classes_, lik, score):
    print(f"(c) P(email|{c}) = {l:.4e}   (d) score = {s:.4e}")
print("    classified as:", m1.predict(x_new)[0], "| posterior:", m1.predict_proba(x_new).round(3)[0])

### Q2 Bernoulli NB: cafeteria reviews, new review {good, tasty, rude, clean}

In [ ]:
revs = [("tasty clean good friendly", "Positive"), ("slow dirty rude bad", "Negative"),
        ("clean fast tasty good", "Positive"), ("rude bad dirty slow", "Negative"),
        ("clean good friendly tasty", "Positive"), ("dirty bland slow bad", "Negative")]
vocab2 = sorted({w for r, _ in revs for w in r.split()})
X2 = np.array([[int(w in r.split()) for w in vocab2] for r, _ in revs])
y2 = np.array([c for _, c in revs])
print("(a) vocabulary (%d):" % len(vocab2), vocab2)
print(X2)
m2 = BernoulliNB(alpha=1.0).fit(X2, y2)
x2 = np.array([[int(w in ("good", "tasty", "rude", "clean")) for w in vocab2]])
print("(b) priors:", dict(zip(m2.classes_, np.exp(m2.class_log_prior_))))
for k, c in enumerate(m2.classes_):
    ll = x2[0] @ np.log(m2.p_[k]) + (1 - x2[0]) @ np.log(1 - m2.p_[k])
    print(f"(c) P(review|{c}) = {np.exp(ll):.4e}")
print("(d) classified as:", m2.predict(x2)[0], "| posterior:", m2.predict_proba(x2).round(4)[0])

### Q3 Gaussian NB: flu diagnosis (sample std, ddof=1, as in the written solution)

In [ ]:
X3 = np.array([[38.5, 110], [39.0, 105], [38.0, 108], [36.5, 80], [36.8, 75], [37.0, 78]])
y3 = np.array(["Flu"] * 3 + ["No Flu"] * 3)
m3 = GaussianNB(var_smoothing=0.0, ddof=1).fit(X3, y3)
print("(a) class       mu_T  sd_T   mu_HR  sd_HR")
for k, c in enumerate(m3.classes_):
    print(f"    {c:7s}  {m3.mu_[k,0]:.3f} {np.sqrt(m3.var_[k,0]):.3f}  {m3.mu_[k,1]:.3f} {np.sqrt(m3.var_[k,1]):.3f}")
x3 = np.array([[37.6, 90.0]])
pdf = lambda x, mu, v: np.exp(-(x - mu) ** 2 / (2 * v)) / np.sqrt(2 * np.pi * v)
for k, c in enumerate(m3.classes_):
    ft, fh = pdf(37.6, m3.mu_[k, 0], m3.var_[k, 0]), pdf(90, m3.mu_[k, 1], m3.var_[k, 1])
    print(f"(b) {c:7s} f(temp)={ft:.4g} f(HR)={fh:.4g}   (c) score={0.5*ft*fh:.3e}")
print("    diagnosis:", m3.predict(x3)[0], "| log-scores:", m3.joint_log_likelihood(x3).round(2)[0])

## 9. Sanity check against scikit-learn (optional)

In [ ]:
try:
    from sklearn.naive_bayes import GaussianNB as G, MultinomialNB as M, BernoulliNB as B
    print("Gaussian    max |dP|:", np.abs(G().fit(Xg[:ntr], yg[:ntr]).predict_proba(Xg[ntr:]) -
                                       GaussianNB().fit(Xg[:ntr], yg[:ntr]).predict_proba(Xg[ntr:])).max())
    print("Multinomial max |dP|:", np.abs(M(alpha=1).fit(Xd[:400], yd[:400]).predict_proba(Xd[400:]) -
                                       mnb.predict_proba(Xd[400:])).max())
    print("Bernoulli   max |dP|:", np.abs(B(alpha=1, binarize=0.5).fit(Xd[:400], yd[:400]).predict_proba(Xd[400:]) -
                                       BernoulliNB(1).fit(Xd[:400] > 0, yd[:400]).predict_proba(Xd[400:])).max())
except Exception as e:
    print("sklearn not available, skipped:", type(e).__name__)

## Summary
* Posterior ~ prior * prod likelihoods; compare classes in log-space.
* Gaussian for continuous, Multinomial for counts, Bernoulli for presence/absence (and it counts absences).
* Laplace smoothing prevents zero probabilities; alpha is a bias/variance dial.
* Independence is false but ranking often survives; probabilities become overconfident.
* Cannot represent interactions (XOR); great fast baseline for sparse, high-dimensional data.